In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv
/kaggle/input/datasets/adityachandra1611/test-traffic/test.csv
/kaggle/input/datasets/adityachandra1611/sample-submission/sample_submission.csv


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor

In [4]:
train = pd.read_csv("/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv")
test = pd.read_csv("/kaggle/input/datasets/adityachandra1611/test-traffic/test.csv")

target = "demand"

X = train.drop(columns=[target])
y = train[target]

X_test = test.copy()

print(X.shape)
print(y.shape)
print(X_test.shape)

(77299, 10)
(77299,)
(41778, 10)


In [5]:
print(train.columns.tolist())
print(train.head())

['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']
   Index geohash  day timestamp    demand     RoadType  NumberofLanes  \
0      0  qp02z1   48       0:0  0.048804          NaN              1   
1      1  qp02zt   48       0:0  0.118507  Residential              3   
2      2  qp08bj   48       0:0  0.027132  Residential              1   
3      3  qp08gt   48       0:0  0.003272  Residential              1   
4      4  qp02zq   48       0:0  0.010819  Residential              1   

  LargeVehicles Landmarks  Temperature Weather  
0   Not Allowed        No          NaN     NaN  
1       Allowed       Yes    31.104565   Sunny  
2   Not Allowed        No    25.919267   Sunny  
3   Not Allowed        No          NaN   Rainy  
4   Not Allowed        No    10.803667   Rainy  


In [6]:
train.drop(
    columns=[
        'year',
        'month',
        'hour',
        'minute',
        'weekofyear'
    ],
    inplace=True,
    errors='ignore'
)

test.drop(
    columns=[
        'year',
        'month',
        'hour',
        'minute',
        'weekofyear'
    ],
    inplace=True,
    errors='ignore'
)

In [7]:
target = 'demand'

X = train.drop(columns=[target])

y = train[target]

X_test = test.copy()

In [8]:
for col in X.columns:

    if X[col].dtype == 'object':

        X[col] = X[col].fillna('Unknown')
        X_test[col] = X_test[col].fillna('Unknown')

    else:

        X[col] = X[col].fillna(X[col].median())
        X_test[col] = X_test[col].fillna(X[col].median())

In [9]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include='object').columns

print(cat_cols)

Index(['geohash', 'timestamp', 'RoadType', 'LargeVehicles', 'Landmarks',
       'Weather'],
      dtype='object')


In [10]:
for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat([
        X[col].astype(str),
        X_test[col].astype(str)
    ])

    le.fit(combined)

    X[col] = le.transform(X[col].astype(str))

    X_test[col] = le.transform(
        X_test[col].astype(str)
    )

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=2000,
    depth=8,
    learning_rate=0.03,
    loss_function='RMSE',
    verbose=100
)

model.fit(X_train, y_train)

pred = model.predict(X_valid)

print(
    "R2 Score:",
    r2_score(y_valid, pred)
)

0:	learn: 0.1389468	total: 67.2ms	remaining: 2m 14s
100:	learn: 0.0650841	total: 789ms	remaining: 14.8s
200:	learn: 0.0612273	total: 1.54s	remaining: 13.8s
300:	learn: 0.0590533	total: 2.3s	remaining: 13s
400:	learn: 0.0571128	total: 3.06s	remaining: 12.2s
500:	learn: 0.0556200	total: 3.81s	remaining: 11.4s
600:	learn: 0.0544861	total: 4.57s	remaining: 10.6s
700:	learn: 0.0535982	total: 5.32s	remaining: 9.85s
800:	learn: 0.0528277	total: 6.07s	remaining: 9.09s
900:	learn: 0.0521612	total: 6.83s	remaining: 8.33s
1000:	learn: 0.0515245	total: 7.64s	remaining: 7.63s
1100:	learn: 0.0508111	total: 8.43s	remaining: 6.88s
1200:	learn: 0.0502920	total: 9.19s	remaining: 6.12s
1300:	learn: 0.0498362	total: 9.95s	remaining: 5.34s
1400:	learn: 0.0493708	total: 10.7s	remaining: 4.58s
1500:	learn: 0.0489892	total: 11.5s	remaining: 3.81s
1600:	learn: 0.0485918	total: 12.2s	remaining: 3.05s
1700:	learn: 0.0482403	total: 13s	remaining: 2.28s
1800:	learn: 0.0478955	total: 13.8s	remaining: 1.52s
1900:	le

In [12]:
print(X.shape)
print(X_test.shape)
print(X.isnull().sum().sum())

(77299, 10)
(41778, 10)
0


In [13]:
CatBoost : 0.50
LightGBM : 0.25
XGBoost : 0.15
ExtraTrees : 0.10

In [14]:
print(X.columns.tolist())

['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']


In [15]:
from sklearn.model_selection import KFold

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [16]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor

In [17]:
cat_model = CatBoostRegressor(
    iterations=1500,
    depth=8,
    learning_rate=0.03,
    verbose=0
)

lgb_model = LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    random_state=42
)

xgb_model = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    random_state=42,
    tree_method="hist"
)

et_model = ExtraTreesRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)

In [18]:
import numpy as np

pred_cat = np.zeros(len(X_test))
pred_lgb = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))
pred_et  = np.zeros(len(X_test))

cv_scores = []

In [19]:
X = X.drop(columns=['Index'])
X_test = X_test.drop(columns=['Index'])

In [20]:
print("Cat :", r2_score(y_val, val_cat))
print("LGB :", r2_score(y_val, val_lgb))
print("XGB :", r2_score(y_val, val_xgb))
print("ET  :", r2_score(y_val, val_et))

NameError: name 'y_val' is not defined

In [21]:
ensemble_pred = (
    0.60 * val_cat +
    0.25 * val_lgb +
    0.10 * val_xgb +
    0.05 * val_et
)

NameError: name 'val_cat' is not defined

In [22]:
pred_cat = np.zeros(len(X_test))
scores = []

for train_idx, val_idx in kf.split(X):

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    cat_model.fit(X_train, y_train)

    val_pred = cat_model.predict(X_val)

    scores.append(r2_score(y_val, val_pred))

    pred_cat += cat_model.predict(X_test)/5

print(np.mean(scores))

0.8527527729932916


In [24]:
print(len(pred_cat))
print(len(pred_lgb))
print(len(pred_xgb))
print(len(pred_et))

41778
41778
41778
41778


In [25]:
final_predictions = (
      0.50 * pred_cat
    + 0.25 * pred_lgb
    + 0.15 * pred_xgb
    + 0.10 * pred_et
)

In [26]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_predictions
})

submission.to_csv("submission.csv", index=False)

In [27]:
print(final_predictions[:5])

[0.03875239 0.01185962 0.01320327 0.01198026 0.02681233]


In [28]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())

   Index    demand
0      0  0.038752
1      1  0.011860
2      2  0.013203
3      3  0.011980
4      4  0.026812


In [29]:
import os
print(os.path.exists("submission.csv"))

True


In [30]:
sub = pd.read_csv("submission.csv")
print(sub.shape)
print(sub.head())

(41778, 2)
   Index    demand
0      0  0.038752
1      1  0.011860
2      2  0.013203
3      3  0.011980
4      4  0.026812


In [31]:
raw_train = pd.read_csv("/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv")
print(raw_train.head())
print(raw_train.columns)

   Index geohash  day timestamp    demand     RoadType  NumberofLanes  \
0      0  qp02z1   48       0:0  0.048804          NaN              1   
1      1  qp02zt   48       0:0  0.118507  Residential              3   
2      2  qp08bj   48       0:0  0.027132  Residential              1   
3      3  qp08gt   48       0:0  0.003272  Residential              1   
4      4  qp02zq   48       0:0  0.010819  Residential              1   

  LargeVehicles Landmarks  Temperature Weather  
0   Not Allowed        No          NaN     NaN  
1       Allowed       Yes    31.104565   Sunny  
2   Not Allowed        No    25.919267   Sunny  
3   Not Allowed        No          NaN   Rainy  
4   Not Allowed        No    10.803667   Rainy  
Index(['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType',
       'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature',
       'Weather'],
      dtype='object')


In [32]:
print(train['timestamp'].nunique())

print(train['timestamp'].value_counts().head(20))

96
timestamp
2:0      1778
1:45     1755
1:30     1750
1:15     1698
1:0      1668
0:30     1617
0:45     1609
0:15     1578
0:0      1534
6:0       928
4:45      925
4:30      923
4:0       921
10:45     920
6:30      917
6:15      917
4:15      914
3:45      914
5:45      913
5:0       911
Name: count, dtype: int64


In [ ]:
print("Unique geohash:", train['geohash'].nunique())

In [ ]:
train = pd.read_csv("/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv")
test = pd.read_csv("/kaggle/input/datasets/adityachandra1611/test-traffic/test.csv")

In [ ]:
for df in [train, test]:

    df['hour'] = df['timestamp'].str.split(':').str[0].astype(int)

    df['minute'] = df['timestamp'].str.split(':').str[1].astype(int)

    df['time_slot'] = (
        df['hour'] * 4 +
        df['minute'] // 15
    )

In [ ]:
for df in [train, test]:

    df['rush_hour'] = (
        ((df['hour'] >= 7) & (df['hour'] <= 10)) |
        ((df['hour'] >= 17) & (df['hour'] <= 20))
    ).astype(int)

In [ ]:
freq = train['geohash'].value_counts()

train['geo_freq'] = train['geohash'].map(freq)

test['geo_freq'] = test['geohash'].map(freq).fillna(0)

In [ ]:
train['road_weather'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['Weather'].astype(str)
)

test['road_weather'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['Weather'].astype(str)
)

In [ ]:
train['road_vehicle'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['LargeVehicles'].astype(str)
)

test['road_vehicle'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['LargeVehicles'].astype(str)
)

In [ ]:
for col in train.columns:

    if train[col].dtype == 'object':

        train[col] = train[col].fillna("Unknown")

        if col in test.columns:
            test[col] = test[col].fillna("Unknown")

    else:

        train[col] = train[col].fillna(
            train[col].median()
        )

        if col in test.columns:
            test[col] = test[col].fillna(
                train[col].median()
            )

In [ ]:
target = "demand"

X = train.drop(columns=[target])

y = train[target]

X_test = test.copy()

In [ ]:
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_weather',
    'road_vehicle'
]

In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

groups = X['geohash']

In [ ]:
model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=10,
    loss_function='RMSE',
    verbose=200
)

In [ ]:
target = "demand"

X = train.drop(columns=[target])
y = train[target]

X_test = test.copy()

In [ ]:
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_weather',
    'road_vehicle'
]

In [ ]:
print(X[cat_features].dtypes)

In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

groups = X['geohash']

In [ ]:
print(train.columns.tolist())

In [ ]:
geo_mean = train.groupby('geohash')['demand'].mean()

train['geo_mean'] = train['geohash'].map(geo_mean)
test['geo_mean'] = test['geohash'].map(geo_mean)

In [ ]:
geo_std = train.groupby('geohash')['demand'].std()

train['geo_std'] = train['geohash'].map(geo_std)
test['geo_std'] = test['geohash'].map(geo_std)

train['geo_std'].fillna(train['geo_std'].median(), inplace=True)
test['geo_std'].fillna(train['geo_std'].median(), inplace=True)

In [ ]:
train['geo_std'] = train['geo_std'].fillna(
    train['geo_std'].median()
)

test['geo_std'] = test['geo_std'].fillna(
    train['geo_std'].median()
)

In [ ]:
print(train['geo_std'].isnull().sum())
print(test['geo_std'].isnull().sum())

In [ ]:
print(train.shape)
print(test.shape)

In [ ]:
print(train.shape)
print(test.shape)

print(train.columns.tolist())

In [ ]:
temp = train.groupby(
    ['geohash','time_slot']
)['demand'].nunique()

print(
    (temp == 1).mean()
)